In [6]:
!pip uninstall -y opencv-python
!pip install opencv-contrib-python

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92


In [11]:
import cv2
import numpy as np

class CNNExtractor:
    """Simulates a CNN (like AlexNet) feature extractor."""
    def extract(self, region):
        # We calculate the standard deviation and mean as a simple 'feature'
        # High contrast regions (like object edges) will have higher scores
        std_val = np.std(region) / 255.0
        mean_val = np.mean(region) / 255.0
        return np.array([std_val, mean_val])

def nms(boxes, scores, overlap_thresh=0.2):
    """Filters out overlapping boxes, keeping only the best ones."""
    if len(boxes) == 0: return np.array([]), np.array([])

    x1, y1 = boxes[:, 0], boxes[:, 1]
    x2, y2 = x1 + boxes[:, 2], y1 + boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        inter = w * h
        ovr = inter / (areas[i] + areas[order[1:]] - inter)

        order = order[np.where(ovr <= overlap_thresh)[0] + 1]

    return boxes[keep], scores[keep]

def get_region_proposals(img):
    """Fallback for Selective Search using a multi-scale grid."""
    h, w = img.shape[:2]
    rects = []
    # R-CNN looks at objects of different sizes (Small, Med, Large)
    for size in [60, 100, 150]:
        for y in range(0, h - size, 40):
            for x in range(0, w - size, 40):
                rects.append([x, y, size, size])
    return np.array(rects)

def rcnn_detect(img, extractor):
    print("Step 1: Generating Region Proposals...")
    try:
        # Attempting 'Smart' Selective Search
        ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
        ss.setBaseImage(img)
        ss.switchToSelectiveSearchFast()
        rects = ss.process()[:200]
        print("   [✓] Selective Search Successful")
    except:
        # Fallback to 'Brute Force' Grid
        rects = get_region_proposals(img)
        print(f"   [!] Using Grid Fallback ({len(rects)} regions)")

    features, valid_boxes = [], []
    print("Step 2: Running Region-based Feature Extraction...")
    for (x, y, w, h) in rects:
        region = img[y:y+h, x:x+w]
        if region.size == 0: continue
        features.append(extractor.extract(region))
        valid_boxes.append([x, y, w, h])

    # Step 3: Scoring (Objects usually have more 'texture' than background)
    features = np.array(features)
    # std_val is the first element; high std = likely object
    scores = features[:, 0] * 10
    scores = 1.0 / (1.0 + np.exp(-scores + 2)) # Normalize to 0-1

    # Step 4: Refine with NMS
    print("Step 3: Scoring & Non-Maximum Suppression...")
    mask = scores > 0.65 # Confidence threshold
    return nms(np.array(valid_boxes)[mask], scores[mask])

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Create test image with two objects
    canvas = np.ones((400, 600, 3), dtype=np.uint8) * 240
    # Rectangle
    cv2.rectangle(canvas, (100, 100), (250, 250), (180, 100, 50), -1)
    # Circle
    cv2.circle(canvas, (450, 200), 70, (50, 150, 50), -1)

    model = CNNExtractor()
    final_boxes, final_scores = rcnn_detect(canvas, model)

    # Draw result
    for (x, y, w, h), score in zip(final_boxes, final_scores):
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(canvas, f"Score: {score:.2f}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imwrite("final_rcnn_output.jpg", canvas)
    print("\n[DONE] Successfully processed image. Check 'final_rcnn_output.jpg'")

Step 1: Generating Region Proposals...
   [!] Using Grid Fallback (314 regions)
Step 2: Running Region-based Feature Extraction...
Step 3: Scoring & Non-Maximum Suppression...

[DONE] Successfully processed image. Check 'final_rcnn_output.jpg'
